# 07b — Cible ancrée sur la date de l'image (+ âge) — 112 participants

**Pourquoi ce notebook existe.** Dans le notebook 07, la cible `progression` compare le *tout premier*
diagnostic du patient au dernier. Mais l'image utilisée (IRM + PET) est prise en moyenne 2 ans plus tard
(jusqu'à 11 ans). Résultat : des patients déjà aggravés au moment de l'image sont étiquetés
« va s'aggraver ». On demande alors au modèle de *prédire* quelque chose qui est déjà arrivé.

**Ce que fait 07b (le notebook 07 n'est pas modifié) :**

1. fixe la date de référence **T0 = date de l'image** (`date_prediction` du manifeste des 112) ;
2. définit `progression_image` = « le diagnostic devient **pire** qu'au moment de l'image, lors d'une visite
   **postérieure** à T0 » ;
3. prend les variables du tableau **à la date de l'image** (et jamais après la visite de départ) ;
4. calcule l'**âge** à T0 (il était vide dans le notebook 07) ;
5. signale les patients **déjà déments** à T0 (stade maximal : ils ne peuvent pas s'aggraver).

Sortie : un nouveau dossier `cible_ancree_image_<date>/jeu_modelisation_112_ancre.csv`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

# Manifeste des 112 : une ligne par participant, avec l'IRM, le PET et le diagnostic de depart.
base = pd.read_csv(C.BASE_112, dtype={"PTID": "string"},
                   parse_dates=["date_irm", "date_pet", "date_dx", "date_prediction"])
base["PTID"] = base["PTID"].str.strip().str.upper()
ptids = set(base["PTID"])
print("Participants :", len(base), "| uniques :", base["PTID"].nunique())

# T0 = date de prediction du manifeste (= la plus tardive des deux images).
base["T0"] = base["date_prediction"]
print("Ecart IRM -> T0 (jours) : min", (base["T0"] - base["date_irm"]).dt.days.min(),
      "| max", (base["T0"] - base["date_irm"]).dt.days.max())
print("Ecart PET -> T0 (jours) : min", (base["T0"] - base["date_pet"]).dt.days.min(),
      "| max", (base["T0"] - base["date_pet"]).dt.days.max())

## 1. Historique des diagnostics (même nettoyage que le notebook 07)

Codes : 1 = CN (normal), 2 = MCI (trouble léger), 3 = démence. Les jours où un même patient a deux codes
contradictoires sont écartés.

In [ ]:
dx = pd.read_csv(C.table("All_Subjects_DXSUM_"), dtype="string", encoding="utf-8-sig")
dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(dx["EXAMDATE"].str.strip(), format="%Y-%m-%d", errors="coerce")
dx = dx.loc[dx["PTID"].isin(ptids) & dx["DIAGNOSIS"].isin(["1", "2", "3"]) & dx["date_dx"].notna(),
            ["PTID", "date_dx", "DIAGNOSIS"]]

conflits = dx.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
conflits = conflits[conflits > 1].index
idx = pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]])
dx = dx.loc[~idx.isin(conflits)].drop_duplicates(["PTID", "date_dx"]).copy()
dx["code"] = dx["DIAGNOSIS"].astype(int)
print("Visites diagnostiques retenues :", len(dx), "| participants :", dx["PTID"].nunique())

## 2. La nouvelle cible

- **Diagnostic de départ** `dx_image` = diagnostic proche de l'IRM (≤ 30 jours), déjà choisi dans le manifeste.
- **Visites futures** = visites diagnostiques **strictement postérieures à T0 + 30 jours**. Le délai de 30 jours
  garantit qu'on ne recompte pas la visite de départ (le diagnostic de départ est à ≤ 30 jours de l'IRM).
- `progression_image = 1` si le **pire** diagnostic futur est **plus grave** que `dx_image`, sinon 0.
- Sans aucune visite future, la cible est inconnue (NaN) : on ne devine pas.

In [ ]:
MARGE_JOURS = 30

lignes = []
for r in base.itertuples(index=False):
    h = dx.loc[dx["PTID"] == r.PTID]
    futur = h.loc[h["date_dx"] > r.T0 + pd.Timedelta(days=MARGE_JOURS)]
    dx_image = int(r.DIAGNOSIS)
    lignes.append({
        "PTID": r.PTID,
        "date_reference": r.T0,
        "dx_image": dx_image,
        "n_visites_futures": len(futur),
        "dx_futur_max": int(futur["code"].max()) if len(futur) else np.nan,
        "suivi_apres_image_jours": int((futur["date_dx"].max() - r.T0).days) if len(futur) else np.nan,
    })
cibles = pd.DataFrame(lignes)
cibles["PTID"] = cibles["PTID"].astype("string")
cibles["progression_image"] = np.where(
    cibles["dx_futur_max"].notna(), (cibles["dx_futur_max"] > cibles["dx_image"]).astype(float), np.nan)
cibles["deja_dement"] = (cibles["dx_image"] == 3).astype(int)

print("Sans visite future (cible inconnue) :", int(cibles["progression_image"].isna().sum()))
print("\nDiagnostic a la date de l'image :")
print(cibles["dx_image"].map({1: "CN", 2: "MCI", 3: "Demence"}).value_counts().to_string())
print("\nProgression apres l'image (tous les 112) :")
print(cibles["progression_image"].value_counts(dropna=False).to_string())
print("\nSuivi apres l'image (annees) :")
print((cibles["suivi_apres_image_jours"] / 365.25).describe().round(1).to_string())

# Comparaison avec l'ancienne cible du notebook 07.
jeu07 = pd.read_csv(sorted(C.COHORTE.glob("preparation_modele_*"))[-1] / "jeu_modelisation_359.csv",
                    dtype={"PTID": "string"})
cmp = cibles.merge(jeu07[["PTID", "progression", "dx_debut"]], on="PTID", how="left")
print("\nAncienne cible (notebook 07) vs nouvelle :")
display(pd.crosstab(cmp["progression"], cmp["progression_image"], rownames=["ancienne"],
                    colnames=["nouvelle"]))
print("Patients dont l'etiquette change :",
      int((cmp["progression"] != cmp["progression_image"]).sum()), "/", len(cmp))

## 3. Variables du tableau, mesurées à la date de l'image

Règle : pour chaque variable, on prend la mesure **la plus récente à T0 + 30 jours ou avant** (jamais une mesure
plus tardive : elle pourrait refléter l'évolution qu'on cherche à prédire).
Tolérance en arrière : 180 jours pour les scores cognitifs, 365 jours pour les biomarqueurs.

In [ ]:
ref = cibles[["PTID", "date_reference"]].copy()
ref["borne"] = ref["date_reference"] + pd.Timedelta(days=MARGE_JOURS)


def mesure_a_t0(prefixe, col_date, colonnes, tolerance_jours):
    """Mesure la plus recente a T0+30j ou avant, dans la tolerance donnee."""
    t = pd.read_csv(C.table(prefixe), dtype="string", encoding="utf-8-sig")
    t["PTID"] = t["PTID"].str.strip().str.upper()
    t["date"] = pd.to_datetime(t[col_date].str.strip(), errors="coerce").dt.normalize()
    noms = list(colonnes.values())
    for brut, nom in colonnes.items():
        t[nom] = pd.to_numeric(t[brut], errors="coerce")
    t = (t.loc[t["PTID"].isin(ptids)].dropna(subset=["PTID", "date"])[["PTID", "date"] + noms]
          .dropna(subset=noms, how="all").drop_duplicates(["PTID", "date"]).sort_values("date"))
    lien = pd.merge_asof(ref.sort_values("borne"), t, by="PTID", left_on="borne", right_on="date",
                         direction="backward", tolerance=pd.Timedelta(days=tolerance_jours))
    return lien[["PTID"] + noms]


tab = cibles[["PTID"]].copy()
for prefixe, cd, cols, tol in [
    ("All_Subjects_MMSE_", "VISDATE", {"MMSCORE": "MMSE_debut"}, 180),
    ("All_Subjects_CDR_", "VISDATE", {"CDGLOBAL": "CDR_global", "CDRSB": "CDR_SB"}, 180),
    ("All_Subjects_ADAS_", "VISDATE", {"TOTAL13": "ADAS13"}, 180),
    ("All_Subjects_FAQ_", "VISDATE", {"FAQTOTAL": "FAQ_debut"}, 180),
    ("All_Subjects_UPENNBIOMK_ROCHE_", "EXAMDATE", {"ABETA42": "ABETA42", "TAU": "TAU", "PTAU": "PTAU"}, 365),
]:
    tab = tab.merge(mesure_a_t0(prefixe, cd, cols, tol), on="PTID", how="left", validate="one_to_one")

# Demographie : sexe, etudes, annee de naissance -> AGE A T0.
def premier(s):
    s = s.str.strip().replace("", pd.NA).dropna()
    return s.iloc[0] if len(s) else pd.NA

demo = pd.read_csv(C.table("All_Subjects_PTDEMOG_"), dtype="string", encoding="utf-8-sig")
demo["PTID"] = demo["PTID"].str.strip().str.upper()
d = demo.loc[demo["PTID"].isin(ptids)].groupby("PTID").agg(
    sexe=("PTGENDER", premier), education=("PTEDUCAT", premier), naissance=("PTDOBYY", premier)).reset_index()
d["annee_naissance"] = pd.to_numeric(d["naissance"].astype("string").str.extract(r"(\d{4})")[0], errors="coerce")
d["sexe"] = pd.to_numeric(d["sexe"], errors="coerce")
d["education"] = pd.to_numeric(d["education"], errors="coerce")
tab = tab.merge(d.drop(columns="naissance"), on="PTID", how="left", validate="one_to_one")
tab = tab.merge(cibles[["PTID", "date_reference"]], on="PTID")
# Age = annee de T0 - annee de naissance (seule l'annee de naissance est fournie par ADNI).
tab["age_reference"] = tab["date_reference"].dt.year - tab["annee_naissance"]

# APOE : nombre d'alleles e4.
apoe = pd.read_csv(C.table("All_Subjects_APOERES_"), dtype="string", encoding="utf-8-sig")
apoe["PTID"] = apoe["PTID"].str.strip().str.upper()
apoe = apoe.loc[apoe["PTID"].isin(ptids)].groupby("PTID")["GENOTYPE"].agg(premier).rename("APOE_genotype").reset_index()
apoe["APOE_e4"] = apoe["APOE_genotype"].map(
    lambda g: pd.NA if pd.isna(g) or "/" not in str(g) else sum(a.strip() == "4" for a in str(g).split("/"))
).astype("Int64")
tab = tab.merge(apoe, on="PTID", how="left", validate="one_to_one").drop(columns="date_reference")

remplissage = pd.DataFrame({"valeurs": tab.notna().sum(), "taux_%": (tab.notna().mean() * 100).round(1)})
display(remplissage.sort_values("taux_%", ascending=False))
print("Age min / median / max :", tab["age_reference"].min(), tab["age_reference"].median(), tab["age_reference"].max())

## 4. Assemblage et enregistrement

On garde aussi l'ancienne cible (`progression_ancienne`) et la cible MMSE à 1 an, pour pouvoir comparer
avant / après correction.

In [ ]:
jeu = (cibles.merge(tab, on="PTID", validate="one_to_one")
             .merge(jeu07[["PTID", "progression", "dx_debut"]].rename(
                 columns={"progression": "progression_ancienne", "dx_debut": "dx_debut_ancien"}),
                 on="PTID", how="left", validate="one_to_one")
             .merge(base[["PTID", "cible_MMSE_1an"]], on="PTID", how="left", validate="one_to_one"))
jeu["progression_ancienne"] = jeu["progression_ancienne"].astype("Int64")

SORTIE = C.sortie("cible_ancree_image")
jeu.to_csv(SORTIE / "jeu_modelisation_112_ancre.csv", index=False, encoding="utf-8-sig")
print("Jeu :", jeu.shape, "| fichier :", SORTIE / "jeu_modelisation_112_ancre.csv")

n_inconnu = int(jeu["progression_image"].isna().sum())
print("\nPopulation 'a risque' (pas encore dements a T0) :",
      int(((jeu["deja_dement"] == 0) & jeu["progression_image"].notna()).sum()), "patients")
print("   dont progressent :",
      int(((jeu["deja_dement"] == 0) & (jeu["progression_image"] == 1)).sum()))
print("Deja dements a T0 (ne peuvent pas s'aggraver) :", int(jeu["deja_dement"].sum()))
print("Cible inconnue :", n_inconnu)
display(jeu.head())